In [10]:
import pandas as pd
import numpy as np

from sklearn.metrics import (
    classification_report,
    accuracy_score
)


In [11]:
TARGET = "Condition_Class"
STATES = ["Excellent", "Good", "Fair", "Poor"]

# Load datasets
train = pd.read_csv("../training_image_features.csv")
test = pd.read_csv("../testing_image_features.csv")

# Sort samples within each image
def ordered_groups(df):
    df = df.copy()

    df["_sample_num"] = pd.to_numeric(
        df["Sample_ID"].astype(str).str.extract(r"(\d+)$")[0],
        errors="coerce"
    )

    for _, group in df.groupby("Image_ID", sort=True):
        yield group.sort_values("_sample_num")

In [12]:
# HMM implementation
class DiscreteHMM:

    def __init__(self, states, n_symbols, smoothing=0.5):
        self.states = states
        self.n_states = len(states)
        self.n_symbols = n_symbols

        self.state_index = {
            state: i for i, state in enumerate(states)
        }

        self.start = np.full(self.n_states, smoothing)

        self.transition = np.full(
            (self.n_states, self.n_states), smoothing
        )

        self.emission = np.full(
            (self.n_states, n_symbols), smoothing
        )

    def fit(self, sequences):

        for labels, observations in sequences:

            if len(labels) == 0:
                continue

            states = [
                self.state_index[label] for label in labels
            ]

            # Initial state probabilities
            self.start[states[0]] += 1

            # Transition probabilities
            for a, b in zip(states[:-1], states[1:]):
                self.transition[a, b] += 1

            # Emission probabilities
            for state, obs in zip(states, observations):
                self.emission[state, obs] += 1

        self.start /= self.start.sum()

        self.transition /= self.transition.sum(
            axis=1, keepdims=True
        )

        self.emission /= self.emission.sum(
            axis=1, keepdims=True
        )

        return self

    def predict(self, observations):

        observations = np.asarray(observations, dtype=int)

        n = len(observations)
        S = self.n_states

        log_start = np.log(self.start)
        log_transition = np.log(self.transition)
        log_emission = np.log(self.emission)

        dp = np.full((n, S), -np.inf)
        back = np.zeros((n, S), dtype=int)

        # Initialization
        dp[0] = (
            log_start
            + log_emission[:, observations[0]]
        )

        # Recursion
        for t in range(1, n):

            scores = (
                dp[t - 1][:, None]
                + log_transition
            )

            back[t] = np.argmax(scores, axis=0)

            dp[t] = (
                np.max(scores, axis=0)
                + log_emission[:, observations[t]]
            )

        # Termination
        path = np.zeros(n, dtype=int)
        path[-1] = np.argmax(dp[-1])

        # Backtracking
        for t in range(n - 2, -1, -1):
            path[t] = back[t + 1, path[t + 1]]

        return np.array(self.states)[path]

In [13]:
damage = pd.to_numeric(
    train["Damage_Score"], errors="coerce"
)

median_damage = damage.median()
damage = damage.fillna(median_damage)

cut_points = np.unique(
    np.quantile(damage, [0.25, 0.50, 0.75])
)

n_symbols = len(cut_points) + 1


def get_observations(group):

    scores = pd.to_numeric(
        group["Damage_Score"], errors="coerce"
    ).fillna(median_damage)

    return np.digitize(scores, cut_points)


# Prepare training sequences
training_sequences = []

for group in ordered_groups(train):

    labels = group[TARGET].astype(str).tolist()
    observations = get_observations(group)

    training_sequences.append(
        (labels, observations)
    )


# Train HMM
hmm = DiscreteHMM(
    STATES,
    n_symbols
)

hmm.fit(training_sequences)


In [14]:
# Create observation symbols from Damage_Score
damage = pd.to_numeric(
    train["Damage_Score"], errors="coerce"
)

median_damage = damage.median()
damage = damage.fillna(median_damage)

cut_points = np.unique(
    np.quantile(damage, [0.25, 0.50, 0.75])
)

n_symbols = len(cut_points) + 1


def get_observations(group):

    scores = pd.to_numeric(
        group["Damage_Score"], errors="coerce"
    ).fillna(median_damage)

    return np.digitize(scores, cut_points)


# Prepare training sequences
training_sequences = []

for group in ordered_groups(train):

    labels = group[TARGET].astype(str).tolist()
    observations = get_observations(group)

    training_sequences.append(
        (labels, observations)
    )


# Train HMM
hmm = DiscreteHMM(
    STATES,
    n_symbols
)

hmm.fit(training_sequences)

# Predict test sequences
y_true = []
y_pred = []

for group in ordered_groups(test):

    labels = group[TARGET].astype(str).tolist()
    observations = get_observations(group)

    predictions = hmm.predict(observations)

    y_true.extend(labels)
    y_pred.extend(predictions.tolist())


# Classification metrics
print("Hidden Markov Model")
print("Accuracy:", accuracy_score(y_true, y_pred))

print(classification_report(
    y_true,
    y_pred,
    labels=STATES,
    zero_division=0
))

Hidden Markov Model
Accuracy: 0.8998138957816377
              precision    recall  f1-score   support

   Excellent       0.00      0.00      0.00        41
        Good       0.95      1.00      0.97       716
        Fair       0.92      0.96      0.94      2193
        Poor       0.48      0.31      0.38       274

    accuracy                           0.90      3224
   macro avg       0.59      0.57      0.57      3224
weighted avg       0.87      0.90      0.89      3224

